# Neural-Network Manhours Model — Crew Day Batching

## Step 1 — Setup

### 1a. Imports and the shared function module

In [1]:
import copy
import os
import sys
import time
import importlib

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score

# Where the shared module lives. Adjust MODULE_DIR if your folder layout differs.
NOTEBOOK_DIR = os.getcwd()
MODULE_DIR = os.environ.get("VEG_MODULE_DIR",
                            os.path.join(os.path.abspath(os.path.join(NOTEBOOK_DIR, os.pardir)), "src", "modeling"))
for path in (MODULE_DIR, NOTEBOOK_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

# Prefer the bundled module (shipped next to this notebook); fall back to the updated copy in src/modeling.
for module_name in ("veg_pipeline_functions_v2", "veg_pipeline_updated"):
    try:
        vp = importlib.import_module(module_name)
        print(f"Shared functions loaded from: {vp.__file__}")
        break
    except ModuleNotFoundError:
        continue
else:
    raise ModuleNotFoundError(f"Could not find veg_pipeline_updated.py / veg_pipeline_functions_v2.py in {MODULE_DIR}")

Shared functions loaded from: /mnt/projects/VEG_Contractors/2026unitization/2026unitization/src/modeling/veg_pipeline_functions_v2.py


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1b. Configuration
Everything you might change lives here.

In [2]:
# ---- Phase switch --------------------------------------------------------------------------
READY_FOR_FINAL_HOLDOUT = False   # <-- BREAKPOINT: True only for the one-time final holdout run
RUN_MODE = os.environ.get("VEG_RUN_MODE", "full")   # "full" | "fast" (quick smoke test)
FAST = RUN_MODE == "fast"
SHOW_PLOTS = True                 # figures are always saved to FIG_DIR too

# ---- Paths ---------------------------------------------------------------------------------
OUTPUT_DIR = os.environ.get("VEG_OUTPUT_DIR", "./deployment")
INPUT_DIR = os.environ.get("VEG_INPUT_DIR", "../input_data")
FULL_SPLIT_DIR = os.path.join(OUTPUT_DIR, "splits", "full_features")   # written by the CatBoost pipeline
DECISIONS_PATH = os.path.join(OUTPUT_DIR, "final_decisions.json")      # tiers, area column, sample weights
NN_DIR = os.path.join(OUTPUT_DIR, "nn")
FIG_DIR = os.path.join(NN_DIR, "figures")
TB_DIR = os.path.join(NN_DIR, "tensorboard")                          # TensorBoard event files
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TB_DIR, exist_ok=True)

# ---- Protective devices ----------------------------------------------------------------------
DEVICE_SOURCE = os.path.join(INPUT_DIR, "span_data.pkl")   # needs a bay column + device columns
DEVICE_BAY_COL = "bay"
PROTECTIVE_DEVICE_COLS = ["protective_device", "major_protective_device"]

# ---- TensorBoard -----------------------------------------------------------------------------
USE_TENSORBOARD = True
TB_PORT = 6006                    # first port to try; the next 10 are tried if it is busy
TB_HOST = "localhost"             # use "0.0.0.0" to reach it from another machine
TB_LOG_TRIALS = True              # one TensorBoard run per tuning trial (useful, but many runs)

# ---- Network / training settings -------------------------------------------------------------
NN_CFG = {
    "n_trials": 3 if FAST else 30,          # hyperparameter search size
    "max_epochs": 15 if FAST else 300,      # hard cap; early stopping usually ends sooner
    "patience": 4 if FAST else 25,          # epochs without improvement before stopping
    "es_feeder_share": 0.15,                # share of fitting feeders held out for early stopping
    "n_seeds_final": 1 if FAST else 5,      # models averaged for the final predictions
    "min_category_count": 20,               # rarer categories are treated as "unknown"
    "skew_log_threshold": 2.0,              # log-transform numeric columns this skewed
    "clip_z": 6.0,                          # clip standardized values to +/- this
    "w_span_mae": 0.5,                      # selection score = 0.5 * span MAE + 0.5 * feeder WAPE
    "w_feeder_wape": 0.5,                   #   (each relative to a naive predictor)
    "quantile_alphas": (0.10, 0.90),        # P10 / P90 heads
    "quantile_loss_weight": 0.5,            # weight of the P10/P90 terms vs the point loss
    "use_chosen_area_only": True,           # drop the three area columns the pipeline rejected
    "weight_scheme": "primary",             # tier weights: "primary" | "hours" | "volume" | None
    "run_group_permutation": True,
    "tune": "auto",                         # True | False | "auto" (re-tune only if inputs changed)
    "train_mae_sample": 5000,               # rows used to track training error each epoch
}
LOSS_CHOICES = ["l1", "smooth_l1"]          # l1 == MAE; smooth_l1 is the Huber-style alternative
ARCH_CHOICES = ["shared_two_head", "split"] # one network with 2 heads, or 2 separate networks
DEFAULT_SPEC = {"arch": "shared_two_head", "loss": "l1", "width": 256, "n_blocks": 3, "dropout": 0.15,
                "emb_dropout": 0.05, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 512, "weight_strength": 0.0}

print(f"RUN_MODE={RUN_MODE} | READY_FOR_FINAL_HOLDOUT={READY_FOR_FINAL_HOLDOUT}")
pd.Series(NN_CFG)

RUN_MODE=full | READY_FOR_FINAL_HOLDOUT=False


n_trials                         30
max_epochs                      300
patience                         25
es_feeder_share                0.15
n_seeds_final                     5
min_category_count               20
skew_log_threshold              2.0
clip_z                          6.0
w_span_mae                      0.5
w_feeder_wape                   0.5
quantile_alphas          (0.1, 0.9)
quantile_loss_weight            0.5
use_chosen_area_only           True
weight_scheme               primary
run_group_permutation          True
tune                           auto
train_mae_sample               5000
dtype: object

### 1c. Reproducibility and hardware

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = vp.RANDOM_STATE


def set_seed(seed):
    """Same seed -> same shuffling, dropout masks and initial weights."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fig_path(name):
    return os.path.join(FIG_DIR, name)


set_seed(SEED)
print(f"Device: {DEVICE} | torch {torch.__version__} | seed {SEED}")

Device: cpu | torch 2.1.2 | seed 42


## Step 2 — TensorBoard
Every training run writes its learning curves to `TB_DIR`. The cell below starts a TensorBoard server, prints its
address, and embeds the dashboard in the notebook when possible. Refresh the dashboard as training cells run.

**What to look at in TensorBoard**
- `loss/*`: the training objective and its parts (point loss, P10 loss, P90 loss).
- `mae/train` vs `mae/early_stop` vs `mae/validation`: a widening gap means overfitting.
- `interval/*`: whether the P10–P90 band covers about 80% of spans.
- `optim/lr`: when the learning rate was cut.
- `optim/grad_norm`: spikes point to unstable training.
- The HPARAMS tab compares tuning trials.

In [4]:
SummaryWriter = None
if USE_TENSORBOARD:
    try:
        from torch.utils.tensorboard import SummaryWriter
    except ImportError:
        print("tensorboard is not installed (pip install tensorboard) - curves will be shown with matplotlib only.")
        USE_TENSORBOARD = False

TB_URL = None

import subprocess

def launch_tensorboard_external(logdir, port, host):
    command = [
        "tensorboard",
        f"--logdir={logdir}",
        f"--port={port}",
        f"--host={host}",
        "--reload_interval=15"
    ]
    process = subprocess.Popen(command)
    return f"http://{host}:{port}", process.pid

def launch_tensorboard(logdir, port, host):
    """Start TensorBoard in the background and return its URL (tries 10 ports)."""
    from tensorboard import program
    last_error = None
    for p in range(port, port + 10):
        try:
            tb = program.TensorBoard()
            tb.configure(argv=[None, "--logdir", logdir, "--port", str(p), "--host", host, "--reload_interval", "15"])
            return tb.launch(), p
        except Exception as err:          # port in use, etc.
            last_error = err
    raise RuntimeError(f"Could not start TensorBoard: {last_error}")

if USE_TENSORBOARD:
    TB_URL, TB_PID = launch_tensorboard_external(os.path.abspath(TB_DIR), TB_PORT, TB_HOST)
    print("=" * 72)
    print(f"TensorBoard is running at: {TB_URL} (PID: {TB_PID})")
    print(f"Log directory: {os.path.abspath(TB_DIR)}")


def make_writer(run_name):
    """One TensorBoard run per training run. Returns None when TensorBoard is off."""
    if not USE_TENSORBOARD:
        return None
    return SummaryWriter(log_dir=os.path.join(TB_DIR, run_name))

TensorBoard is running at: http://localhost:6006 (PID: 2692151)
Log directory: /mnt/projects/VEG_Contractors/2026unitization/2026unitization/notebooks/deployment/nn/tensorboard


## Step 3 — Load data and decisions

### 3a. Feature splits
These files were produced by the CatBoost pipeline (Section 7) and hold every feature for all four splits.
Feeders never appear in more than one split.

In [5]:
FULL_X, FULL_Y, FULL_META, MANIFEST = vp.load_full_splits(FULL_SPLIT_DIR)

split_overview = pd.DataFrame({
    s: {"spans": len(FULL_X[s]), "feeders": FULL_META[s]["feeder"].nunique(),
        "inaccessible_share": FULL_META[s]["inaccessible"].mean(),
        "mean_hours": FULL_Y[s].mean(), "median_hours": FULL_Y[s].median(), "p90_hours": FULL_Y[s].quantile(0.9)}
    for s in ["train", "valid", "test", "holdout"]}).T
vp.show(split_overview, "Split overview", 3)

# No feeder may appear in two splits (otherwise scores are optimistic).
for a, b in [("train", "valid"), ("train", "test"), ("train", "holdout"),
             ("valid", "test"), ("valid", "holdout"), ("test", "holdout")]:
    assert set(FULL_META[a]["feeder"]).isdisjoint(FULL_META[b]["feeder"]), f"feeder overlap {a}/{b}"
print("Feeder separation check passed.")


Split overview


,spans,feeders,inaccessible_share,mean_hours,median_hours,p90_hours
train,12337.0,249.0,0.457,4.267,3.0,8.0
valid,2182.0,54.0,0.480,3.856,3.0,8.0
test,4159.0,54.0,0.469,4.777,4.0,10.0
holdout,4671.0,73.0,0.376,3.862,3.0,8.0


Feeder separation check passed.


In [6]:
# Group samples by `crew_leader` and `complete_date`, include null
df = vp.load_span_data(INPUT_DIR)
grouped_df = df.groupby(["crew_leader", "complete_date"], dropna=False)

# Assign crew-day grouping a unique ID for easy batching
df["crew_day_id"] = grouped_df.ngroup()

Duplicate bay records found: 36
Spans: 24,284 | unique completion days: 134


In [7]:
# Join `FULL_META` with splits using `bay_unique` for batching
for key in FULL_META:
    FULL_META[key] = FULL_META[key].merge(df[["bay_unique", "crew_day_id"]], on="bay_unique", how="left")  # Adjust columns as needed

In [8]:
FULL_META["train"].columns

Index(['bay_unique', 'feeder', 'inaccessible', 'impactful_issues', 'veg_tier',
       'bay', 'area_class__dominant_land_cover_general', 'area_class__area_lc',
       'area_class__area_lcxfm', 'area_class__area_xfmr', 'crew_day_id'],
      dtype='object')

In [9]:
class NeuralNet(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=256):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.15)
        self.activation = nn.ReLU()
    
    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.dropout(x)
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        return x

# Instantiate the model
input_dim = FULL_X["train"].shape[1]  # Number of features
output_dim = 1  # Predicting total manhours
model = NeuralNet(input_dim=input_dim, output_dim=output_dim)

In [10]:
FULL_X["train"].columns

Index(['bay_blue_count', 'bay_blue_volume_(cubic_ft)', 'bay_green_count',
       'bay_green_volume_(cubic_ft)', 'bay_l&r_blue_(ft)',
       'bay_l&r_green_(ft)', 'bay_l&r_orange_(ft)', 'bay_l&r_purple_ft',
       'bay_l&r_red_(ft)', 'bay_l&r_yellow_(ft)',
       ...
       'road_feature_source_missing', 'road_distance_tier',
       'road_access_category', 'nearest_road_fc_name',
       'dominant_land_cover_type', 'dominant_land_cover_general', 'area_lc',
       'area_lcxfm', 'area_xfmr', 'heavy_contact_conductor'],
      dtype='object', length=136)

In [11]:
FULL_META["train"].columns

Index(['bay_unique', 'feeder', 'inaccessible', 'impactful_issues', 'veg_tier',
       'bay', 'area_class__dominant_land_cover_general', 'area_class__area_lc',
       'area_class__area_lcxfm', 'area_class__area_xfmr', 'crew_day_id'],
      dtype='object')

In [12]:
from torch.utils.tensorboard import SummaryWriter

# Set up TensorBoard
writer = SummaryWriter(log_dir=TB_DIR)
writer.add_text("Config", str(NN_CFG))  # Log hyperparameters for reference

In [13]:
from sklearn.preprocessing import LabelEncoder

# Optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=DEFAULT_SPEC["lr"], weight_decay=DEFAULT_SPEC["weight_decay"])
loss_fn = nn.L1Loss()

def apply_label_encoding(df, categorical_cols):
    """
    Convert categorical features into numerical using LabelEncoder.
    """
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))  # Convert all values to string for consistent format
        label_encoders[col] = le  # Save encoders for future use
    return df, label_encoders


def group_batches(meta, features, targets):
    """
    Create batches by grouping data based on 'crew_day_id',
    while ensuring 'crew_day_id' is not included in the features.
    """
    meta = meta[['bay_unique', 'crew_day_id']]  # Only include necessary columns
    grouped_meta = meta.groupby("crew_day_id", dropna=False)  # Group by crew_day_id
    
    categorical_cols = features.select_dtypes(include=["object", "category"]).columns
    if len(categorical_cols) > 0:
        features, _ = apply_label_encoding(features, categorical_cols)  # Label encoding for categorical features
    
    batched_data = []
    for crew_day_id, group in grouped_meta:
        # Join features and targets using 'bay_unique'
        group_data = (
            group[['bay_unique']].merge(features, on="bay_unique", how="left")
                                 .merge(targets, on="bay_unique", how="left")
        )
        
        # Exclude the crew_day_id from inputs
        input_features = group_data.columns.difference(["crew_day_id", "bay_unique", "manhours"]).tolist()
        
        # Prepare input and target tensors
        inputs = torch.tensor(group_data[input_features].values, dtype=torch.float32)
        outputs = torch.tensor(group_data["manhours"].values, dtype=torch.float32)
        
        # Add batch
        batched_data.append(TensorDataset(inputs, outputs))
    
    return batched_data


# Prepare DataLoader
train_batched = group_batches(FULL_META["train"], FULL_X["train"], FULL_Y["train"])
train_loader = DataLoader(train_batched, batch_size=DEFAULT_SPEC["batch_size"], shuffle=True)


epochs = NN_CFG["max_epochs"]

for epoch in range(epochs):
    model.train()  # Set model to training mode
    epoch_losses = []

    for batch in train_loader:
        inputs, targets = batch
        optimizer.zero_grad()  # Reset gradients
        outputs = model(inputs)  # Forward pass
        loss = loss_fn(outputs.squeeze(), targets)  # Compute loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        epoch_losses.append(loss.item())
    
    # Log training metrics to TensorBoard
    mean_loss = np.mean(epoch_losses)  # Average training loss for the epoch
    writer.add_scalar("Loss/Train", mean_loss, epoch)
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {mean_loss:.4f}")


Traceback (most recent call last):
  File "/usr/local/bin/tensorboard", line 8, in <module>
    sys.exit(run_main())
  File "/usr/local/lib/python3.10/dist-packages/tensorboard/main.py", line 39, in run_main
    tensorboard = program.TensorBoard(plugins=default.get_plugins())
  File "/usr/local/lib/python3.10/dist-packages/tensorboard/default.py", line 83, in get_plugins
    return get_static_plugins() + get_dynamic_plugins()
  File "/usr/local/lib/python3.10/dist-packages/tensorboard/default.py", line 118, in get_dynamic_plugins
    return [
  File "/usr/local/lib/python3.10/dist-packages/tensorboard/default.py", line 119, in <listcomp>
    entry_point.load()
  File "/usr/lib/python3.10/importlib/metadata/__init__.py", line 171, in load
    module = import_module(match.group('module'))
  File "/usr/lib/python3.10/importlib/__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "<frozen importlib._bootstrap>", line 1050, in _gcd_

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'torch.utils.data.dataset.TensorDataset'>

In [ ]:
# Label Encoding for Validation Set Features
# Preprocessing Validation Features (Reusing Train Functions)
#categorical_cols = FULL_X["valid"].select_dtypes(include=["object", "category"]).columns
#FULL_X["valid"], _ = apply_label_encoding(FULL_X["valid"], categorical_cols)  # Label Encoding

# Group validation batches (Reuse group_batches function from training)
valid_batched = group_batches(FULL_META["valid"], FULL_X["valid"], FULL_Y["valid"])

# Create DataLoader for validation
valid_loader = DataLoader(valid_batched, batch_size=DEFAULT_SPEC["batch_size"])

# Validation Loop
model.eval()
valid_losses = []
valid_predictions = []
valid_targets = []

with torch.no_grad():  # No gradient required for validation
    for batch in valid_loader:
        inputs, targets = batch
        predictions = model(inputs).squeeze()  # Inference

        # Calculate validation loss
        loss = loss_fn(predictions, targets)
        valid_losses.append(loss.item())

        # Collect predictions and targets
        valid_predictions.extend(predictions.cpu().numpy())
        valid_targets.extend(targets.cpu().numpy())

# Metrics Computation
mean_valid_loss = np.mean(valid_losses)
valid_r2_score = r2_score(valid_targets, valid_predictions)

print(f"Validation MAE: {mean_valid_loss:.4f}")
print(f"Validation R² Score: {valid_r2_score:.4f}")

# Log validation metrics to TensorBoard
writer.add_scalar("Loss/Valid", mean_valid_loss, epochs)
writer.add_scalar("R2/Valid", valid_r2_score, epochs)
print(f"Epoch {epoch+1}/{epochs} | Valid Loss: {mean_valid_loss:.4f} | Valid R²: {valid_r2_score:.4f}")
writer.close()

In [ ]:
from tensorboard import notebook

# Start TensorBoard
notebook.start("--logdir ./deployment/nn/tensorboard")